confirm model.track() gives you per-frame track_id + bbox + keypoints. Nothing more. D

In [1]:
from ultralytics import YOLO

model = YOLO("best.pt")   # your trained YOLO11n-pose

results = model.track(
    source="mmavid1.mp4",   # trimmed clip from step 2 (or full fight for now)
    tracker="botsort.yaml",
    persist=True,
    stream=True,        # generator — don't load whole video into memory
    device="mps",       # fall back to "cpu" if mps errors
    verbose=False,
)

for i, r in enumerate(results):
    if r.boxes is None or r.boxes.id is None:
        continue
    ids = r.boxes.id.int().tolist()        # track IDs this frame
    n_kpts = r.keypoints.shape[1] if r.keypoints is not None else 0
    print(f"frame {i}: ids={ids} kpts_per_obj={n_kpts}")
    if i >= 20:        # just sanity-check first ~20 frames
        break

frame 0: ids=[1, 2] kpts_per_obj=17
frame 1: ids=[1, 2] kpts_per_obj=17
frame 2: ids=[1, 2] kpts_per_obj=17
frame 3: ids=[1, 2] kpts_per_obj=17
frame 4: ids=[1, 2] kpts_per_obj=17
frame 5: ids=[1, 2] kpts_per_obj=17
frame 6: ids=[1, 2] kpts_per_obj=17
frame 7: ids=[1, 2] kpts_per_obj=17
frame 8: ids=[1, 2] kpts_per_obj=17
frame 9: ids=[1, 2] kpts_per_obj=17
frame 10: ids=[1, 2] kpts_per_obj=17
frame 11: ids=[1, 2] kpts_per_obj=17
frame 12: ids=[1, 2] kpts_per_obj=17
frame 13: ids=[1, 2] kpts_per_obj=17
frame 14: ids=[1, 2] kpts_per_obj=17
frame 15: ids=[1, 2] kpts_per_obj=17
frame 16: ids=[1, 2] kpts_per_obj=17
frame 17: ids=[1, 2] kpts_per_obj=17
frame 18: ids=[1, 2] kpts_per_obj=17
frame 19: ids=[1, 2] kpts_per_obj=17
frame 20: ids=[1, 2] kpts_per_obj=17


This code will show through which frames is the model able to track 2 distinct fighters and how many labels it produces.

Heartbeat (live IDs): watch this scroll. If live IDs keeps showing new high numbers ([1,2] → [1,7] → [15,2]), you're literally watching IDs get dropped and re-acquired in real time. That's the fragmentation happening.

unique IDs total: the headline diagnostic. A clean 2-fighter clip should be ~2–4. The higher this is, the worse the tracking.

top 8 tracks by frame count: this is the most useful one. It shows you how a fighter fragmented: if you see one fighter as ID 2: 2100 frames but the other smeared across ID 1: 800, ID 7: 600, ID 15: 700, that's one real fighter split into three. Tells you exactly which IDs to merge if you later want to manually stitch a segment.

In [2]:
from collections import Counter
from ultralytics import YOLO

def run_and_pair(weights, source, device="mps", log_every=200):
    model = YOLO(weights)
    results = model.track(
        source=source, tracker="botsort.yaml",
        persist=True, stream=True, device=device, verbose=False,
    )

    frames = []
    counts = Counter()
    all_ids_seen = set()
    frames_with_2 = 0

    for i, r in enumerate(results):
        frame = {}
        if r.boxes is not None and r.boxes.id is not None:
            ids = r.boxes.id.int().tolist()
            boxes = r.boxes.xywh.cpu().numpy()
            kpts = r.keypoints.data.cpu().numpy()
            for tid, box, kp in zip(ids, boxes, kpts):
                frame[tid] = (box, kp)
                counts[tid] += 1
                all_ids_seen.add(tid)
        frames.append(frame)
        if len(frame) == 2:
            frames_with_2 += 1

        # heartbeat: shows progress + what IDs are live right now
        if i % log_every == 0:
            live = sorted(frame.keys())
            print(f"frame {i:>5} | live IDs: {live} | "
                  f"unique IDs so far: {len(all_ids_seen)} | "
                  f"both-fighter frames: {frames_with_2}")

    total = len(frames)
    fighter_ids = [tid for tid, _ in counts.most_common(2)]

    # end-of-run summary — the important part
    print("\n=== TRACKING SUMMARY ===")
    print(f"total frames:           {total}")
    print(f"unique track IDs total: {len(all_ids_seen)}   <- high = fragmentation")
    print(f"locked fighter IDs:     {fighter_ids}")
    print(f"both fighters present:  {frames_with_2}/{total} "
          f"({100*frames_with_2/total:.0f}%)")
    print("\ntop 8 tracks by frame count:")
    for tid, c in counts.most_common(8):
        print(f"  ID {tid:>3}: {c:>5} frames ({100*c/total:.0f}% of clip)")

    return fighter_ids, frames

In [3]:
fighter_ids, frames = run_and_pair("best.pt", "mmavid2.mp4")

frame     0 | live IDs: [1, 2] | unique IDs so far: 2 | both-fighter frames: 1
frame   200 | live IDs: [1, 2] | unique IDs so far: 2 | both-fighter frames: 201
frame   400 | live IDs: [1, 2] | unique IDs so far: 2 | both-fighter frames: 401
frame   600 | live IDs: [1, 2] | unique IDs so far: 2 | both-fighter frames: 601
frame   800 | live IDs: [1, 2] | unique IDs so far: 2 | both-fighter frames: 801
frame  1000 | live IDs: [2, 15] | unique IDs so far: 4 | both-fighter frames: 994
frame  1200 | live IDs: [2, 15] | unique IDs so far: 4 | both-fighter frames: 1194
frame  1400 | live IDs: [15, 16] | unique IDs so far: 5 | both-fighter frames: 1373
frame  1600 | live IDs: [15, 16] | unique IDs so far: 5 | both-fighter frames: 1573
frame  1800 | live IDs: [15, 16] | unique IDs so far: 5 | both-fighter frames: 1773

=== TRACKING SUMMARY ===
total frames:           1802
unique track IDs total: 5   <- high = fragmentation
locked fighter IDs:     [2, 15]
both fighters present:  1774/1802 (98%)



see where exactly the delination point occurs where more than 2 unique ids are labeled to the clip

In [4]:
import cv2
cap = cv2.VideoCapture("mmavid2.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print("fps:", fps, "-> frame 850 =", 850/fps, "seconds")

fps: 29.964341298227666 -> frame 850 = 28.36705107381339 seconds


trim the clip to the exact second

In [5]:
!ffmpeg -i mmavid2.mp4 -t 28 -c copy mmavid2trim.mp4

ffmpeg version 8.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 14.0.0 (clang-1400.0.29.202)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60. 26.100 / 60. 26.100
  libavcodec     62. 28.100 / 62. 28.100
  libavformat    62. 12.100 / 62. 12.100
  libavdevice    62.  3.100 / 62.  3.100
  libavfilter    11. 14.100 / 11. 14.100
  libswscale      9.  5.100 /  9.  5.100
  libswresample   6.  3.100 /  6.  3.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'mmavid2.mp4':
  Metadata:
    major_brand     : mp42
    minor_version   : 1
    compatible_brands: isommp41mp42
    creation_time   : 2026-06-20T10:57:27.000000Z
  

OSError: [Errno 5] Input/output error

reconfirming the're are only 2 fighters within the trimmed clip

In [ ]:
fighter_ids, frames = run_and_pair("best.pt", "mmavid2trim.mp4")

frame     0 | live IDs: [1, 2] | unique IDs so far: 2 | both-fighter frames: 1
frame   200 | live IDs: [1, 2] | unique IDs so far: 2 | both-fighter frames: 201
frame   400 | live IDs: [1, 2] | unique IDs so far: 2 | both-fighter frames: 401
frame   600 | live IDs: [1, 2] | unique IDs so far: 2 | both-fighter frames: 601
frame   800 | live IDs: [1, 2] | unique IDs so far: 2 | both-fighter frames: 801

=== TRACKING SUMMARY ===
total frames:           841
unique track IDs total: 2   <- high = fragmentation
locked fighter IDs:     [1, 2]
both fighters present:  841/841 (100%)

top 8 tracks by frame count:
  ID   1:   841 frames (100% of clip)
  ID   2:   841 frames (100% of clip)


based on the COCO keypoints and the geometery of ankles work we can determine if a fighter is southpaw or orthodox in a particular frame

In [ ]:
import numpy as np

# COCO-17 indices
L_ANK, R_ANK = 15, 16
L_KNE, R_KNE = 13, 14
L_HIP, R_HIP = 11, 12

# (left_idx, right_idx) pairs, most reliable first
LR_PAIRS = [(L_ANK, R_ANK), (L_KNE, R_KNE), (L_HIP, R_HIP)]

def frame_stance(kpts_self, kpts_opp, min_vis=0.5, vote_pairs=LR_PAIRS):
    """
    kpts_self, kpts_opp: (17,3) arrays of (x, y, vis).
    Returns 'orthodox' | 'southpaw' | 'ambiguous'.
    Lead foot = limb further toward the opponent along x.
    """
    self_cx = kpts_self[:, 0].mean()
    opp_cx  = kpts_opp[:, 0].mean()
    forward = np.sign(opp_cx - self_cx)   # +1 opp is to the right, -1 to the left
    if forward == 0:
        return "ambiguous"

    votes = []
    for li, ri in vote_pairs:
        lx, lv = kpts_self[li, 0], kpts_self[li, 2]
        rx, rv = kpts_self[ri, 0], kpts_self[ri, 2]
        if lv < min_vis or rv < min_vis:
            continue                       # occluded limb, skip this pair
        # which side is more "forward" (toward opponent)?
        if forward > 0:                    # opponent to the right -> larger x leads
            lead_left = lx > rx
        else:                              # opponent to the left -> smaller x leads
            lead_left = lx < rx
        votes.append(lead_left)            # True = left leads = orthodox

    if not votes:
        return "ambiguous"                 # nothing visible enough to decide
    left_votes = sum(votes)
    right_votes = len(votes) - left_votes
    if left_votes == right_votes:
        return "ambiguous"
    return "orthodox" if left_votes > right_votes else "southpaw"

We are going to try an alternate method that also takes into account the fighter's ankles, hips, shoulder and nose to get more accurate reading of stances esp in weird angles.

here we are testing on frame 120 of the izzy vs whittaker 1 fight where they are in an openstance matchup and izzy is in southpaw while bobby knuckles is in orthodox

In [7]:
def parse_yolo_pose_line(line):
    p = list(map(float, line.split()))
    kpts = np.array(p[5:]).reshape(17, 3)   # skip class + bbox
    cx = p[1]
    return cx, kpts

with open("120.txt") as f:
    rows = [parse_yolo_pose_line(l) for l in f if l.strip()]

# order by cx so we know who's left/right
rows.sort(key=lambda r: r[0])
(cxA, kA), (cxB, kB) = rows[0], rows[1]
print("left fighter :", frame_stance(kA, kB))   # expect orthodox
print("right fighter:", frame_stance(kB, kA))   # expect southpaw

left fighter : orthodox
right fighter: southpaw


- first i am going to create a list for each fighter for each frame what stance they are in
- then I am going to use the smoothing time frame to kill any misreported or occulded stance switches (robert fumbles or trips so his stance is all over the place but hes fr an orthodox fighter)
- then take all smoothed frames and combine them and output the actual held stance (A stance only becomes 'official' once it holds >= min_hold consecutive frames.)
- run a summary for a fighter for a full clip that includes they majority stance, how many times they swtiched and when exactly in the clip they swtich stances

In [8]:
import numpy as np
from collections import deque, Counter

# adjustable knobs
SMOOTH_WIN = 11    # frames for glitch-removal majority vote (~0.3s @30fps). keep small + odd.
MIN_SWITCH_FRAMES = 35   # how long a new stance must HOLD to count as a real switch. 30 = 1s @30fps.
FPS = fps                 # for converting frame indices to timestamps in the output


def per_frame_stances(frames, fighter_ids):
    """raw stance label per fighter per frame.
    Returns {fighter_id: [label_or_None per frame]}. None = fighter absent that frame."""
    out = {fid: [] for fid in fighter_ids}
    a, b = fighter_ids
    for frame in frames:
        for self_id, opp_id in ((a, b), (b, a)):
            if self_id in frame and opp_id in frame:
                _, k_self = frame[self_id]
                _, k_opp = frame[opp_id]
                out[self_id].append(frame_stance(k_self, k_opp))
            else:
                out[self_id].append(None)   # can't compute without both
    return out


def smooth(labels, win=SMOOTH_WIN):
    """sliding majority vote to kill single-frame flicker.
    None and 'ambiguous' don't vote; if no real votes in window, output None."""
    half = win // 2
    sm = []
    for i in range(len(labels)):
        window = labels[max(0, i-half): i+half+1]
        votes = [x for x in window if x in ("orthodox", "southpaw")]
        if not votes:
            sm.append(None)
        else:
            sm.append(Counter(votes).most_common(1)[0][0])
    return sm


def segment_switches(smoothed, min_hold=MIN_SWITCH_FRAMES):
    """collapse smoothed labels into held stance segments.
    A stance only becomes 'official' once it holds >= min_hold consecutive frames.
    Returns list of (stance, start_frame, end_frame)."""
    segments = []
    current = None          # the last officially-confirmed stance
    run_label = None        # stance of the current candidate run
    run_start = 0
    for i, lab in enumerate(smoothed):
        if lab != run_label:
            run_label = lab
            run_start = i
        # has this run held long enough AND is it a real stance?
        if (lab in ("orthodox", "southpaw")
                and i - run_start + 1 >= min_hold
                and lab != current):
            segments.append((lab, run_start, i))
            current = lab
    return segments


def summarize(frames, fighter_ids,
              smooth_win=SMOOTH_WIN, min_switch=MIN_SWITCH_FRAMES, fps=FPS):
    """full per-fighter report."""
    raw = per_frame_stances(frames, fighter_ids)
    report = {}
    for fid in fighter_ids:
        smoothed = smooth(raw[fid], smooth_win)
        counts = Counter(x if x else "absent" for x in smoothed)
        total = len(smoothed)
        segs = segment_switches(smoothed, min_switch)
        report[fid] = {
            "pct": {k: round(100*v/total) for k, v in counts.items()},
            "switches": max(0, len(segs) - 1),   # first segment = initial stance, not a switch
            "timeline": [(s, f"{a/fps:.1f}s", f"{b/fps:.1f}s") for s, a, b in segs],
        }
    return report

In [9]:
report = summarize(frames, fighter_ids, smooth_win=SMOOTH_WIN, min_switch=MIN_SWITCH_FRAMES, fps=FPS)
for fid, r in report.items():
    print(f"\n--- fighter {fid} ---")
    print("stance %:", r["pct"])
    print("confirmed switches:", r["switches"])
    for stance, start, end in r["timeline"]:
        print(f"   {stance:9} confirmed at {start} (held through {end})")


--- fighter 2 ---
stance %: {'absent': 74, 'southpaw': 5, 'orthodox': 21}
confirmed switches: 2
   orthodox  confirmed at 29.9s (held through 31.0s)
   southpaw  confirmed at 33.4s (held through 34.6s)
   orthodox  confirmed at 35.7s (held through 36.9s)

--- fighter 15 ---
stance %: {'absent': 74, 'southpaw': 14, 'orthodox': 12}
confirmed switches: 1
   southpaw  confirmed at 29.3s (held through 30.4s)
   orthodox  confirmed at 38.4s (held through 39.6s)


In [ ]:
re-reads the clip for pixels (the frames list has keypoints, not image data — expected, not the desync bug), redraws each fighter's skeleton, and writes a live stance label above each box. Output is an .mp4 you scrub through

In [10]:
import cv2
import numpy as np

# COCO-17 skeleton edges for drawing
SKELETON = [
    (5,7),(7,9),(6,8),(8,10),          # arms
    (11,13),(13,15),(12,14),(14,16),   # legs
    (5,6),(11,12),(5,11),(6,12),       # torso
    (0,5),(0,6),                       # head-to-shoulders
]

def per_frame_smoothed(frames, fighter_ids, smooth_win):
    """Recompute the smoothed per-frame stance the overlay will display
    (same pipeline summarize uses, exposed per-frame)."""
    raw = per_frame_stances(frames, fighter_ids)
    return {fid: smooth(raw[fid], smooth_win) for fid in fighter_ids}

def make_overlay(clip_path, frames, fighter_ids, out_path,
                 smooth_win=SMOOTH_WIN, min_vis=0.5):
    sm = per_frame_smoothed(frames, fighter_ids, smooth_win)

    cap = cv2.VideoCapture(clip_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(out_path,
                             cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    # stable color per fighter ID (BGR)
    colors = {fighter_ids[0]: (0, 200, 0), fighter_ids[1]: (0, 120, 255)}
    stance_color = {"orthodox": (0, 200, 0), "southpaw": (0, 120, 255), None: (160,160,160)}

    fi = 0
    while True:
        ok, img = cap.read()
        if not ok or fi >= len(frames):
            break
        frame = frames[fi]
        for fid in fighter_ids:
            if fid not in frame:
                continue
            box, kp = frame[fid]          # box = cx,cy,w,h ; kp = (17,3) x,y,conf
            col = colors[fid]

            # skeleton
            for a, c in SKELETON:
                if kp[a,2] >= min_vis and kp[c,2] >= min_vis:
                    pa = (int(kp[a,0]), int(kp[a,1]))
                    pc = (int(kp[c,0]), int(kp[c,1]))
                    cv2.line(img, pa, pc, col, 2)
            for j in range(17):
                if kp[j,2] >= min_vis:
                    cv2.circle(img, (int(kp[j,0]), int(kp[j,1])), 3, col, -1)

            # stance label above the box
            cx, cy, w, h = box
            x1, y1 = int(cx - w/2), int(cy - h/2)
            label = sm[fid][fi]
            txt = f"F{fid}: {(label or 'amb').upper()}"
            lc = stance_color.get(label, (160,160,160))
            cv2.rectangle(img, (x1, y1-22), (x1+170, y1), (0,0,0), -1)
            cv2.putText(img, txt, (x1+4, y1-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, lc, 2)

        # frame counter + timestamp, top-left
        cv2.putText(img, f"frame {fi}  {fi/fps:.1f}s",
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        writer.write(img)
        fi += 1

    cap.release()
    writer.release()
    print(f"wrote {fi} frames -> {out_path}")

In [11]:
make_overlay("mmavid2trim.mp4", frames, fighter_ids,
             "stance_overlaymaybenotgood.mp4", smooth_win=9)

wrote 841 frames -> stance_overlaymaybenotgood.mp4
